In [ ]:
# Importação da biblioteca generativa do Google
import google.generativeai as genai
# Importação de módulos do Google Colab para manipulação de dados
from google.colab import userdata, output

In [ ]:
# Importação de módulos do Flask para criar uma API web
from flask import Flask, request, jsonify

In [ ]:
# Importação de módulos do Flask-CORS para lidar com problemas de CORS
from flask_cors import CORS

In [ ]:
# Importação do módulo ngrok
from pyngrok import ngrok

In [ ]:
# Importação de módulos do Python para manipulação de arquivos e diretórios
import os
import json

In [ ]:
# Configuração do ambiente de execução
API_Key_Name = "saude"
# Verifica se a chave de API está disponível no ambiente de execução do Google Colab
if userdata.get(API_Key_Name) is None:
    # Se não estiver disponível, tenta carregar a chave de API do arquivo de configuração
    try:
        with open(f"/content/{API_Key_Name}.json") as f:
            api_data = json.load(f)
            api_key = api_data.get(API_Key_Name)  # Obtém a chave de API do arquivo JSON
    except FileNotFoundError:
        raise ValueError(f"API key not found. Please set the '{API_Key_Name}' in userdata or provide a {API_Key_Name}.json file.")
else:
    # Se a chave de API estiver disponível, carrega o valor diretamente do ambiente do Google Colab
    gemini_API_Key = userdata.get(API_Key_Name)

# Configuração da chave de API do Google Gemini
genai.configure(api_key=gemini_API_Key)

In [ ]:
# Inicialização do modelo de linguagem generativa
model = genai.GenerativeModel("models/gemini-1.5-pro-latest")

In [ ]:
# Inicialização do Flask
app = Flask(__name__)

In [ ]:
# Permitir CORS para todas as origens
CORS(app)

In [ ]:
def gerar_diagnostico(sintomas, nivel_Sintoma, observacoes_enfermeiro=""):
    prompt = f"""
    Avalie o paciente e estabeleça uma conduta com termos técnicos e sucintos.

    **Sintomas:** {sintomas}
    **Nível do Sintoma (0-10):** {nivel_Sintoma}
    **Observações do Enfermeiro:** {observacoes_enfermeiro}
    """
    chat = model.start_chat()  # Inicia um novo chat para cada diagnóstico
    resposta = chat.send_message(prompt)
    return resposta.text

In [ ]:
@app.route('/api/diagnostico', methods=['POST'])
def api_diagnostico():
    data = request.get_json()
    sintomas = data.get('sintomas')
    nivel_Sintoma = data.get('nivel_Sintoma')
    observacoes_enfermeiro = data.get('observacoes_enfermeiro', "")  # Valor padrão se não for fornecido
    diagnostico = gerar_diagnostico(sintomas, nivel_Sintoma, observacoes_enfermeiro)
    return jsonify({'diagnostico': diagnostico})


In [ ]:
if __name__ == '__main__':
    # Configurar o authtoken do ngrok
    # Verificar se o authtoken do ngrok está disponível no ambiente de execução
    if "Auth_ngrok" not in os.environ:
        # Se não estiver disponível, tenta carregar o authtoken do arquivo de configuração
        try:
            with open("/content/ngrok_authtoken.txt") as f:
                ngrok_API_Key = f.read().strip()
        except FileNotFoundError:
            raise ValueError("ngrok authtoken not found. Please set the 'ngrok_authtoken' environment variable or provide a ngrok_authtoken.txt file.")
    else:
        # Se o authtoken estiver disponível, carrega o valor diretamente do ambiente
        ngrok_API_Key = os.environ["Auth_ngrok"]
    ngrok_API_Key = userdata.get("Auth_ngrok")
    ngrok.set_auth_token(ngrok_API_Key)
    ngrok_tunnel = ngrok.connect(5000)# Cria um túnel ngrok para a porta 5000 (onde o Flask app estará rodando).
    print('Endereço público da API:', ngrok_tunnel.public_url)
    app.run(host='0.0.0.0', port=5000)# Executa o Flask app, tornando-o acessível em todos os endereços IP (0.0.0.0) na porta 5000